# SD 1.5 Inpainting · RemovalBench

In [ ]:
import hashlib
import json
import os
import zipfile
from pathlib import Path

project = "sd15"
archive_name = "replaceme-sd15-env.zip"
expected_manifest = "50d92487030fcdab3558d5f47588a9be1ef90601a057bfdda9350fe01c8924ca"
kaggle = Path("/kaggle/input").is_dir()
default_input = Path("/kaggle/input/replaceme-sd15-env") if kaggle else Path.cwd()
inputs = Path(os.environ.get("REPLACEME_INPUT", default_input)).expanduser().resolve()
work = Path(os.environ.get("REPLACEME_WORK", "/kaggle/working" if kaggle else "replaceme-work")).resolve()

archive_path = inputs if inputs.name == archive_name else inputs / archive_name
if archive_path.is_file():
    bundle = archive_path
    with zipfile.ZipFile(bundle) as archive:
        metadata = archive.read("environment.json")
        bootstrap = archive.read("bootstrap.py")
elif inputs.is_dir() and (inputs / "environment.json").is_file():
    bundle = inputs
    metadata = (bundle / "environment.json").read_bytes()
    bootstrap = (bundle / "bootstrap.py").read_bytes()
else:
    raise FileNotFoundError(f"Ожидается {archive_path}. Укажите точный путь к ZIP или каталогу Dataset в REPLACEME_INPUT.")

if hashlib.sha256(metadata).hexdigest() != expected_manifest:
    raise ValueError(f"Манифест не соответствует комплекту {archive_name}.")
manifest = json.loads(metadata)
if manifest["project"] != project or hashlib.sha256(bootstrap).hexdigest() != manifest["bootstrap_sha256"]:
    raise ValueError("Ноутбук и загрузчик окружения относятся к разным сборкам.")
scope = {}
exec(compile(bootstrap, "bootstrap.py", "exec"), scope)
scope["start"](bundle, manifest, work / project)


## Загрузка весов и данных

In [ ]:
%%benchmark
"""Fetch the two pinned public releases and verify every file against the Hub tree."""
import fcntl
import hashlib
import json
import random
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.request import urlopen

MODEL = "stable-diffusion-v1-5/stable-diffusion-inpainting"
MODEL_REV = "8a4288a76071f7280aedbdb3253bdb9e9d5d84bb"
DATA = "BaiLing/RemovalBench"
DATA_REV = "6dcd0fbe7d81755fb132f2fe8c27edfc602b9965"
GROUPS = ("images", "masks", "gt", "results_omnieraser_base")


def tree(kind, repo, revision):
    url = f"https://huggingface.co/api/{kind}/{repo}/revision/{revision}?blobs=true"
    with urlopen(url, timeout=60) as response:
        info = json.load(response)
    assert info["sha"] == revision
    return {item["rfilename"]: item for item in info["siblings"]}


def valid(path, meta):
    if not path.is_file() or path.stat().st_size != meta["size"]:
        return False
    if "lfs" in meta:
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest() == meta["lfs"]["sha256"]
    content = path.read_bytes()
    return hashlib.sha1(f"blob {len(content)}\0".encode() + content).hexdigest() == meta["blobId"]


def fetch(job):
    url, path, meta = job
    if valid(path, meta):
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    part = path.with_suffix(path.suffix + ".part")
    subprocess.run(["curl", "-fsSL", "--retry", "4", "--retry-all-errors", "--connect-timeout", "20",
                    "--continue-at", "-", "--speed-limit", "1024", "--speed-time", "90",
                    "-o", str(part), url], check=True)
    if not valid(part, meta):
        part.unlink()
        raise ValueError(f"Hub checksum mismatch: {path}")
    part.replace(path)
    print(path, flush=True)


def download(root):
    manifest = root / "sample.json"
    ready = root / ".assets-verified.json"
    if manifest.is_file() and ready.is_file():
        cached = json.loads(ready.read_text())
        sizes = cached.get("files", {})
        if (cached.get("model_revision") == MODEL_REV and cached.get("dataset_revision") == DATA_REV
                and sizes and all((root / name).is_file() and (root / name).stat().st_size == size
                                  for name, size in sizes.items())):
            return
    print("Проверка весов SD 1.5 и примеров RemovalBench на Hugging Face…", flush=True)
    model = tree("models", MODEL, MODEL_REV)
    data = tree("datasets", DATA, DATA_REV)
    common = set.intersection(*({Path(n).stem for n in data if n.startswith(group + "/")
                                 and n.endswith(".png")} for group in GROUPS))
    ids = random.Random(42).sample(sorted(common), 12)
    model_files = [n for n in model if n.endswith(("fp16.safetensors", ".json", ".txt"))
                   or n == "README.md"]
    data_files = ["README.md", *(f"{group}/{i}.png" for i in ids for group in GROUPS)]
    jobs = []
    for repo, rev, meta, names, folder in [
        (MODEL, MODEL_REV, model, model_files, "model"),
        ("datasets/" + DATA, DATA_REV, data, data_files, "removalbench"),
    ]:
        for name in names:
            jobs.append((f"https://huggingface.co/{repo}/resolve/{rev}/{name}",
                         root / folder / name, meta[name]))
    with ThreadPoolExecutor(max_workers=4) as pool:
        list(pool.map(fetch, jobs))
    temp = manifest.with_suffix(".tmp")
    temp.write_text(json.dumps(dict(
        model=MODEL, model_revision=MODEL_REV, dataset=DATA, dataset_revision=DATA_REV,
        population=len(common), selection_seed=42, ids=ids,
    ), indent=2) + "\n")
    temp.replace(manifest)
    checked = dict(model_revision=MODEL_REV, dataset_revision=DATA_REV,
                   files={str(path.relative_to(root)): meta["size"] for _, path, meta in jobs})
    temp.write_text(json.dumps(checked))
    temp.replace(ready)


def main(root):
    root.mkdir(parents=True, exist_ok=True)
    with (root / ".download.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        download(root)



asset_dir = Path.cwd() / "sd15-assets"
main(asset_dir)

## Данные и параметры

In [ ]:
%%benchmark
import csv, json, os, sys, time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import torch
from diffusers import StableDiffusionInpaintPipeline
from IPython.display import display
from PIL import Image, ImageOps

assets = asset_dir
data = assets / "removalbench"
out = Path.cwd() / "removalbench-results"
out.mkdir(exist_ok=True)
sample = json.loads((assets / "sample.json").read_text())
ids = sample["ids"]
prompt, seed, steps, cfg = "There is nothing here.", 24, 28, 7.5
long_edge = 512

def read(folder, key, mode="RGB"):
    with Image.open(data / folder / f"{key}.png") as im:
        return im.convert(mode)

assert torch.cuda.is_available(), "Выберите GPU в Settings → Accelerator."
print(torch.cuda.get_device_name(), "·", len(ids), "из", sample["population"], "·", ", ".join(ids))

## Удаление объекта

In [ ]:
%%benchmark
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    assets / "model", variant="fp16", torch_dtype=torch.float16,
    use_safetensors=True, local_files_only=True,
)
assert pipe.unet.config.in_channels == 9
free_gib = torch.cuda.mem_get_info()[0] / 2**30
offload = free_gib < 4.5
if offload:
    pipe.enable_sequential_cpu_offload()
else:
    pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)
print(type(pipe.scheduler).__name__, "·", "CPU offload" if offload else "CUDA")

In [ ]:
%%benchmark
started = time.monotonic()
for key in ids:
    source, mask = read("images", key), read("masks", key, "L")
    assert source.size == mask.size and mask.getextrema()[0] < 128 <= mask.getextrema()[1]
    size = tuple(8 * max(1, round(side * long_edge / max(source.size) / 8)) for side in source.size)
    result = pipe(
        prompt=prompt, image=source.resize(size, Image.Resampling.LANCZOS),
        mask_image=mask.resize(size, Image.Resampling.NEAREST), width=size[0], height=size[1],
        num_inference_steps=steps, guidance_scale=cfg, strength=1.0,
        generator=torch.Generator("cpu").manual_seed(seed),
    )
    if result.nsfw_content_detected and any(result.nsfw_content_detected):
        raise RuntimeError(f"Изображение {key} отфильтровано моделью.")
    result.images[0].save(out / f"{key}.png")
    print(key, end=" ", flush=True)
elapsed = time.monotonic() - started
print(f" · {elapsed:.0f} с")

## PSNR относительно чистого фона

In [ ]:
%%benchmark
def psnr(error):
    mse = np.mean(error, dtype=np.float64)
    return float(-10 * np.log10(mse)) if mse else float("inf")

rows = []
for key in ids:
    omni = read("results_omnieraser_base", key)
    size = omni.size
    target = np.asarray(read("gt", key).resize(size, Image.Resampling.LANCZOS), dtype=np.float32)
    mask = np.asarray(read("masks", key, "L").resize(size, Image.Resampling.NEAREST)) >= 128
    with Image.open(out / f"{key}.png") as generated:
        for method, img in [("Исходник", read("images", key)), ("SD 1.5", generated), ("OmniEraser Base", omni)]:
            pred = np.asarray(img.resize(size, Image.Resampling.LANCZOS), dtype=np.float32)
            error = ((pred - target) / 255) ** 2
            rows.append(dict(id=key, method=method, full_db=psnr(error), mask_db=psnr(error[mask])))

with (out / "scores.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
summary = {}
print(f"{'Метод':17} {'Кадр, дБ':>9} {'Маска, дБ':>10}")
for method in dict.fromkeys(r["method"] for r in rows):
    group = [r for r in rows if r["method"] == method]
    summary[method] = {k: float(np.mean([r[k] for r in group])) for k in ("full_db", "mask_db")}
    print(f"{method:17} {summary[method]['full_db']:9.2f} {summary[method]['mask_db']:10.2f}")

run = dict(sample=sample, prompt=prompt, seed=seed, steps=steps, guidance_scale=cfg,
           strength=1., long_edge=long_edge, scheduler=type(pipe.scheduler).__name__,
           dtype="float16", offload=offload, gpu=torch.cuda.get_device_name(),
           python=sys.version.split()[0], cuda=torch.version.cuda, seconds=elapsed, mean=summary,
           packages={p: version(p) for p in ("torch", "diffusers", "transformers", "accelerate", "pillow", "numpy")})
(out / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=2, allow_nan=False))

## Исходник · маска · SD 1.5 · OmniEraser Base · чистый фон

In [ ]:
%%benchmark
previews = []
for key in ids[:4]:
    with Image.open(out / f"{key}.png") as generated:
        panels = [read("images", key), read("masks", key), generated, read("results_omnieraser_base", key), read("gt", key)]
        strip = Image.new("RGB", (5 * 192, 192), "white")
        for col, img in enumerate(panels):
            strip.paste(ImageOps.pad(img, (192, 192), color="white"), (192 * col, 0))
    previews.append(strip)
    print(key)
    display(strip)
preview = Image.new("RGB", (960, 192 * len(previews)), "white")
for j, strip in enumerate(previews):
    preview.paste(strip, (0, j * 192))
preview.save(out / "preview.jpg", quality=90)